In [1]:
"""
Ensemble: average of the probabilities of the regularized logistic
regression (run.py) and of the random forest (trees.py).

    p = alpha * p_logistic + (1 - alpha) * p_forest

alpha = 1 -> logistic only, alpha = 0 -> forest only.
alpha and the decision threshold are chosen on the same 80/20 validation
split as all the other experiments.

Usage: python experiments_ensemble.py
"""
import numpy as np

from helpers import create_csv_submission
from preprocessing import split_train_val, best_threshold
from run import load_data, build_features, train, predict_proba
from trees import fit_bins, apply_bins, train_forest, predict_forest

MAKE_SUBMISSION = False   # True -> retrain both models on all data and write a CSV
FOREST = dict(n_trees=50, max_depth=12, min_leaf=50, max_features="sqrt")
ALPHAS = [0.0, 0.25, 0.4, 0.5, 0.6, 0.75, 1.0]


def forest_proba(x_fit, y_fit, x_pred):
    """Train the forest on (x_fit, y_fit) and return its probabilities on x_pred."""
    mean, edges = fit_bins(x_fit)
    forest = train_forest(apply_bins(x_fit, mean, edges), y_fit, verbose=False, **FOREST)
    return predict_forest(forest, apply_bins(x_pred, mean, edges))


def logistic_proba(x_fit, y_fit, x_pred):
    """Train the logistic regression of run.py and return its probabilities on x_pred."""
    w, mean, std = train(x_fit, y_fit)
    return predict_proba(x_pred, w, mean, std)


# ---------------------------------------------------------------------
# Data and split (same as everyone)
# ---------------------------------------------------------------------
np.random.seed(0)
x_train, x_test, y01, test_ids, names = load_data()
x_train, x_test = build_features(x_train, x_test, names)
i_tr, i_val = split_train_val(len(x_train))
y_val = y01[i_val].astype(int)

print("Training the logistic regression...")
p_log = logistic_proba(x_train[i_tr], y01[i_tr], x_train[i_val])
print("Training the random forest (about 1.5 min)...")
p_forest = forest_proba(x_train[i_tr], y01[i_tr].astype(int), x_train[i_val])

# ---------------------------------------------------------------------
# Try several weights
# ---------------------------------------------------------------------
print(f"\n{'alpha':>6} {'F1':>7} {'accuracy':>9} {'threshold':>10}")
best = None
for alpha in ALPHAS:
    p = alpha * p_log + (1 - alpha) * p_forest
    t, f = best_threshold(y_val, p)
    acc = np.mean((p >= t).astype(int) == y_val)
    note = "  (forest only)" if alpha == 0 else "  (logistic only)" if alpha == 1 else ""
    print(f"{alpha:>6.2f} {f:>7.4f} {acc:>9.3f} {t:>10.2f}{note}")
    if best is None or f > best[1]:
        best = (alpha, f, t)

alpha, f_best, t_best = best
print(f"\nBest: alpha = {alpha}, F1 = {f_best:.4f}, threshold = {t_best:.2f}")

# ---------------------------------------------------------------------
# Optional submission: both models retrained on all the training data
# ---------------------------------------------------------------------
if MAKE_SUBMISSION:
    print("\nRetraining on all the data...")
    p_test = (alpha * logistic_proba(x_train, y01, x_test)
              + (1 - alpha) * forest_proba(x_train, y01.astype(int), x_test))
    y_pred = np.where(p_test >= t_best, 1, -1)
    create_csv_submission(test_ids, y_pred, "submission_ensemble.csv")
    print(f"submission_ensemble.csv written, {np.mean(y_pred == 1):.1%} predicted positive")

Training the logistic regression...
Training the random forest (about 1.5 min)...

 alpha      F1  accuracy  threshold
  0.00  0.4258     0.865       0.20  (forest only)
  0.25  0.4300     0.874       0.21
  0.40  0.4309     0.881       0.22
  0.50  0.4301     0.876       0.21
  0.60  0.4303     0.876       0.21
  0.75  0.4289     0.876       0.21
  1.00  0.4279     0.881       0.22  (logistic only)

Best: alpha = 0.4, F1 = 0.4309, threshold = 0.22
